# Project 2: Programmatic Retriever Evaluation

Evaluate whether Chroma retrieves the fable passages needed for each question. Python compares retrieved chunk IDs with the golden IDs and calculates recall and precision. No LLM judge is used.

PDF extraction, 600-character chunking with 100-character overlap, and indexing now live in `create_vectordb.py`. Run it once from the repository root:

```powershell
uv run python 2_fables_retriever_eval/create_vectordb.py
```

When only k changes, reuse the existing database. This notebook loads it without re-extracting or re-embedding the PDF. Select the shared `.venv` kernel and restart the kernel after this refactor.

## Step 1: Open the existing vector database

Run this notebook with either the repository root or this project folder as its working directory. The path setup lets Python import this project's `models` and `tools` modules.

Shared chunk, collection, and embedding settings live in `models/vector_store.py`. Opening the store loads the embedding model for query embeddings; it does not re-index the documents.

In [18]:
import json
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name != "2_fables_retriever_eval":
    project_root = project_root / "2_fables_retriever_eval"

assert project_root.is_dir(), "Run from the repository root or the Project 2 folder."
sys.path.insert(0, str(project_root))

from models.vector_store import load_vector_store
from tools.evaluation import evaluate_retriever
from tools.results import print_results, print_comparison, save_results

vector_store = load_vector_store()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2770.15it/s]


## Step 2: Load golden questions

Send only the question to the retriever. The reference chunk IDs are used afterward by the scorer. The reference answers and excerpts help review labels.

These 24 questions have AI-prepared labels pending human review.

In [19]:
golden_path = project_root / "docs/goldends/goldens.json"
goldens = json.loads(golden_path.read_text(encoding="utf-8"))

print(f"Evaluation questions: {len(goldens)}")

Evaluation questions: 24


## Step 3: Evaluate and save the baseline

The shared evaluator retrieves each question's top k chunks, compares IDs, and records both metrics:

- Recall = matched relevant chunks / all relevant chunks.
- Precision = matched relevant chunks / retrieved chunks.

The overall metrics average the per-question scores. `print_results` displays them; `save_results` writes a separate timestamped report.

In [20]:
baseline_k = 3

baseline = evaluate_retriever(vector_store, goldens, k=baseline_k)
print_results(baseline)
baseline_path = save_results(baseline, label="baseline")

print(f"\nBaseline saved: {baseline_path.name}")

Evaluation at k=3
q001 | Recall: 1.00 | Precision: 0.33
q002 | Recall: 0.67 | Precision: 0.67
q003 | Recall: 1.00 | Precision: 0.33
q004 | Recall: 1.00 | Precision: 0.67
q005 | Recall: 1.00 | Precision: 0.33
q006 | Recall: 0.50 | Precision: 0.33
q007 | Recall: 1.00 | Precision: 0.33
q008 | Recall: 1.00 | Precision: 0.67
q009 | Recall: 1.00 | Precision: 0.33
q010 | Recall: 1.00 | Precision: 0.67
q011 | Recall: 0.50 | Precision: 0.33
q012 | Recall: 0.33 | Precision: 0.33
q013 | Recall: 1.00 | Precision: 0.33
q014 | Recall: 1.00 | Precision: 0.67
q015 | Recall: 1.00 | Precision: 0.67
q016 | Recall: 1.00 | Precision: 0.67
q017 | Recall: 1.00 | Precision: 0.33
q018 | Recall: 1.00 | Precision: 0.67
q019 | Recall: 0.50 | Precision: 0.33
q020 | Recall: 1.00 | Precision: 0.33
q021 | Recall: 0.50 | Precision: 0.33
q022 | Recall: 0.00 | Precision: 0.00
q023 | Recall: 1.00 | Precision: 0.67
q024 | Recall: 0.33 | Precision: 0.33

Questions evaluated: 24
Average Recall@3: 80.56%
Average Precision@3:

## Step 4: Evaluate and save a candidate

Change `candidate_k` to test a different number of results. Reuse the same evaluator, printer, and saver. Keep the golden labels, chunks, and embedding model unchanged.

The candidate is a separate run, so rerunning this cell preserves the baseline and creates a new result file.

In [21]:
candidate_k = 5

candidate = evaluate_retriever(vector_store, goldens, k=candidate_k)
print_results(candidate)
candidate_path = save_results(candidate, label="candidate")

print(f"\nCandidate saved: {candidate_path.name}")

Evaluation at k=5
q001 | Recall: 1.00 | Precision: 0.20
q002 | Recall: 1.00 | Precision: 0.60
q003 | Recall: 1.00 | Precision: 0.20
q004 | Recall: 1.00 | Precision: 0.40
q005 | Recall: 1.00 | Precision: 0.20
q006 | Recall: 0.50 | Precision: 0.20
q007 | Recall: 1.00 | Precision: 0.20
q008 | Recall: 1.00 | Precision: 0.40
q009 | Recall: 1.00 | Precision: 0.20
q010 | Recall: 1.00 | Precision: 0.40
q011 | Recall: 1.00 | Precision: 0.40
q012 | Recall: 1.00 | Precision: 0.60
q013 | Recall: 1.00 | Precision: 0.20
q014 | Recall: 1.00 | Precision: 0.40
q015 | Recall: 1.00 | Precision: 0.40
q016 | Recall: 1.00 | Precision: 0.40
q017 | Recall: 1.00 | Precision: 0.20
q018 | Recall: 1.00 | Precision: 0.40
q019 | Recall: 1.00 | Precision: 0.40
q020 | Recall: 1.00 | Precision: 0.20
q021 | Recall: 1.00 | Precision: 0.40
q022 | Recall: 0.67 | Precision: 0.40
q023 | Recall: 1.00 | Precision: 0.40
q024 | Recall: 0.67 | Precision: 0.40

Questions evaluated: 24
Average Recall@5: 95.14%
Average Precision@5:

## Step 5: Compare results

The comparison helper prints each question's recall plus the overall recall and precision changes. It uses the k values from the runs, so it also works when we try other values.

Miss inspection remains available through `tools.results.print_misses(run)` when needed; this notebook does not run it.

In [22]:
print_comparison(baseline, candidate)

q001 | Recall@3: 1.00 | Recall@5: 1.00
q002 | Recall@3: 0.67 | Recall@5: 1.00
q003 | Recall@3: 1.00 | Recall@5: 1.00
q004 | Recall@3: 1.00 | Recall@5: 1.00
q005 | Recall@3: 1.00 | Recall@5: 1.00
q006 | Recall@3: 0.50 | Recall@5: 0.50
q007 | Recall@3: 1.00 | Recall@5: 1.00
q008 | Recall@3: 1.00 | Recall@5: 1.00
q009 | Recall@3: 1.00 | Recall@5: 1.00
q010 | Recall@3: 1.00 | Recall@5: 1.00
q011 | Recall@3: 0.50 | Recall@5: 1.00
q012 | Recall@3: 0.33 | Recall@5: 1.00
q013 | Recall@3: 1.00 | Recall@5: 1.00
q014 | Recall@3: 1.00 | Recall@5: 1.00
q015 | Recall@3: 1.00 | Recall@5: 1.00
q016 | Recall@3: 1.00 | Recall@5: 1.00
q017 | Recall@3: 1.00 | Recall@5: 1.00
q018 | Recall@3: 1.00 | Recall@5: 1.00
q019 | Recall@3: 0.50 | Recall@5: 1.00
q020 | Recall@3: 1.00 | Recall@5: 1.00
q021 | Recall@3: 0.50 | Recall@5: 1.00
q022 | Recall@3: 0.00 | Recall@5: 0.67
q023 | Recall@3: 1.00 | Recall@5: 1.00
q024 | Recall@3: 0.33 | Recall@5: 0.67

k | Average recall | Average precision
3 | 80.56% | 44.44%
5 | 

## Conclusion

The earlier experiment with this 24-question set produced:

| Setting | Average recall | Average precision |
| --- | ---: | ---: |
| k=3 | 80.56% | 44.44% |
| k=5 | 95.14% | 34.17% |

Increasing k recovered more labeled evidence while returning more irrelevant chunks. We preferred k=5 for that experiment because evidence coverage was the main goal. Use the comparison output above to assess each new run.